In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/bronze/landing/olist"
FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table, file in FILES.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)   # reviews contain line breaks
          .option("escape", '"')
          .csv(f"{BASE}/{file}")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))
    (df.write.mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"workspace.bronze.{table}"))
    print(f"{table}: {df.count():,} rows")